# Salience-Guided Stable Diffusion: Diversity Experiment

Compares standard SD sampling against salience-guided sampling using:
- ScoreNormPhiSD
- DiversityPhiSD

Metrics: pairwise CLIP diversity, T-CLIP

In [1]:
import sys
sys.path.insert(0, "..")
sys.path.insert(0, "../vendor/code/BoN/src_sd")

%load_ext autoreload
%autoreload 2

import torch
import numpy as np
from pathlib import Path
from PIL import Image
from diffusers import DDPMScheduler
from transformers import CLIPModel, CLIPProcessor

from sd.pipeline import SalienceGradSDPipeline
from sd.phi_sd import ScoreNormPhiSD, DiversityPhiSD

DEVICE = "mps" if torch.backends.mps.is_available() else ("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")

MODEL_ID = "runwayml/stable-diffusion-v1-5"
OUTPUTS  = Path("../outputs/sd")
OUTPUTS.mkdir(parents=True, exist_ok=True)

N_PER_PROMPT   = 8
NUM_STEPS      = 500
SALIENCE_SCALE = 1.0


Device: mps


## 1. Load pipelines

In [2]:
from diffusers import StableDiffusionPipeline

# Baseline: standard SD
pipe_base = StableDiffusionPipeline.from_pretrained(
    MODEL_ID, torch_dtype=torch.float16
).to(DEVICE)
pipe_base.scheduler = DDPMScheduler.from_config(
    pipe_base.scheduler.config, timestep_spacing="trailing"
)

# Salience pipeline
pipe_sal = SalienceGradSDPipeline.from_pretrained(
    MODEL_ID, torch_dtype=torch.float16
).to(DEVICE)
pipe_sal.scheduler = DDPMScheduler.from_config(
    pipe_sal.scheduler.config, timestep_spacing="trailing"
)
pipe_sal.set_guidance_frequency(1)
pipe_sal.set_salience_scale(SALIENCE_SCALE)

# Freeze all model parameters
for pipe in [pipe_base, pipe_sal]:
    pipe.vae.requires_grad_(False)
    pipe.text_encoder.requires_grad_(False)
    pipe.unet.requires_grad_(False)

print("Pipelines loaded.")

Loading pipeline components...:   0%|          | 0/7 [00:00<?, ?it/s]

`text_config_dict` is provided which will be used to initialize `CLIPTextConfig`. The value `text_config["id2label"]` will be overriden.


Loading pipeline components...:   0%|          | 0/7 [00:00<?, ?it/s]

`text_config_dict` is provided which will be used to initialize `CLIPTextConfig`. The value `text_config["id2label"]` will be overriden.


Pipelines loaded.


## 2. Null embeddings and prompts

In [4]:
# Null embeddings for ScoreNormPhiSD unconditional mode
with torch.no_grad():
    null_input = pipe_sal.tokenizer(
        "",
        padding="max_length",
        max_length=pipe_sal.tokenizer.model_max_length,
        truncation=True,
        return_tensors="pt",
    )
    null_embeds = pipe_sal.text_encoder(
        null_input.input_ids.to(DEVICE)
    )[0]                                                  # (1, seq_len, dim)

print(f"Null embeds: {null_embeds.shape}")

with open("../prompts/fixed_prompts.txt") as f:
    fixed_prompts = [l.strip() for l in f if l.strip()]
print(f"Loaded {len(fixed_prompts)} prompts")

Null embeds: torch.Size([1, 77, 768])
Loaded 20 prompts


## 3. Baseline sampling

In [6]:
baseline_path = OUTPUTS / "baseline"
baseline_path.mkdir(exist_ok=True)

for prompt in fixed_prompts:
    print(f"Baseline: {prompt[:60]}")
    prompt_dir = baseline_path / prompt
    prompt_dir.mkdir(exist_ok=True)

    result = pipe_base(
        prompt=prompt,
        num_images_per_prompt=N_PER_PROMPT,
        num_inference_steps=NUM_STEPS,
    )

    for idx, img in enumerate(result.images):
        img.save(prompt_dir / f"{idx}.png")

Baseline: a portrait of a person in a crowded city street


  0%|          | 0/500 [00:00<?, ?it/s]

KeyboardInterrupt: 

## 4. ScoreNormPhiSD sampling

In [ ]:
phi_score = ScoreNormPhiSD(conditional=False)
pipe_sal.setup_phi(phi_score, {"null_embeds": null_embeds})
pipe_sal.set_guidance(SALIENCE_SCALE)

score_path = OUTPUTS / "score_norm_phi"
score_path.mkdir(exist_ok=True)
pipe_sal.set_project_path(score_path)

for prompt in fixed_prompts:
    print(f"ScoreNormPhi: {prompt[:60]}")
    pipe_sal(offset=0, prompt=prompt, n_samples=1, block_size=NUM_STEPS,
             num_images_per_prompt=N_PER_PROMPT, num_inference_steps=NUM_STEPS, num_try=1)


## 5. DiversityPhiSD sampling

In [ ]:
phi_div = DiversityPhiSD()
pipe_sal.setup_phi(phi_div, {})
pipe_sal.set_guidance(SALIENCE_SCALE)

div_path = OUTPUTS / "diversity_phi"
div_path.mkdir(exist_ok=True)
pipe_sal.set_project_path(div_path)

for prompt in fixed_prompts:
    print(f"DiversityPhi: {prompt[:60]}")
    pipe_sal(offset=0, prompt=prompt, n_samples=1, block_size=NUM_STEPS,
             num_images_per_prompt=N_PER_PROMPT, num_inference_steps=NUM_STEPS, num_try=1)


## 6. Evaluation

In [ ]:
clip_model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32").to(DEVICE)
clip_proc  = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")
clip_model.eval()

@torch.no_grad()
def clip_embed_images(image_paths):
    images = [Image.open(p).convert("RGB") for p in image_paths]
    inputs = {k: v.to(DEVICE) for k, v in clip_proc(images=images, return_tensors="pt", padding=True).items()}
    feats  = clip_model.get_image_features(**inputs)
    return feats / feats.norm(dim=-1, keepdim=True)

def pairwise_clip_diversity(image_paths):
    feats = clip_embed_images(image_paths).cpu().numpy()
    sim   = feats @ feats.T
    idx   = np.triu_indices(len(feats), k=1)
    return float((1.0 - sim[idx]).mean())

@torch.no_grad()
def t_clip_score(image_paths, prompt):
    images = [Image.open(p).convert("RGB") for p in image_paths]
    inputs = {k: v.to(DEVICE) for k, v in clip_proc(text=[prompt], images=images, return_tensors="pt", padding=True).items()}
    out    = clip_model(**inputs)
    i_feat = out.image_embeds / out.image_embeds.norm(dim=-1, keepdim=True)
    t_feat = out.text_embeds  / out.text_embeds.norm(dim=-1, keepdim=True)
    return float((i_feat @ t_feat.T).mean().cpu())

method_dirs = {"Baseline": baseline_path, "ScoreNormPhi": score_path, "DiversityPhi": div_path}
results = []
for method_name, method_dir in method_dirs.items():
    divs, tclips = [], []
    for prompt in fixed_prompts:
        imgs = sorted((method_dir / prompt).glob("*.png")) if (method_dir / prompt).exists() else []
        if len(imgs) < 2: continue
        divs.append(pairwise_clip_diversity(imgs))
        tclips.append(t_clip_score(imgs, prompt))
    results.append({"method": method_name, "diversity": np.mean(divs), "t_clip": np.mean(tclips)})

print(f"{chr(10)}{"Method":<20} {"CLIP Diversity":>16} {"T-CLIP":>10}")
print("-" * 48)
for r in results:
    print(f"{r[chr(39)]method[chr(39)]:<20} {r[chr(39)]diversity[chr(39)]:>16.4f} {r[chr(39)]t_clip[chr(39)]:>10.4f}")
